# CI/CD Pipeline - Hiring Bias Model
process -> train (local, due to quota) -> evaluate (accuracy + fairness) -> conditional gate -> register/deploy

In [1]:
!pip install "sagemaker==2.219.0" --no-cache-dir --quiet

In [2]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterInteger, ParameterString, ParameterFloat

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()

pipeline_session = PipelineSession()

bucket = "hiring-bias-adult-income-mustafa"  # double check this matches notebook 01
prefix = "hiring-bias-project"

train_s3 = f"s3://{bucket}/{prefix}/data/train/train.csv"
validation_s3 = f"s3://{bucket}/{prefix}/data/validation/validation.csv"
test_s3 = f"s3://{bucket}/{prefix}/data/test/test.csv"

model_package_group_name = "HiringBiasModelPackageGroup"
pipeline_name = "HiringBiasCICDPipeline"

s3_client = boto3.client("s3", region_name=region)

print(region)
print(train_s3)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


us-east-1
s3://hiring-bias-adult-income-mustafa/hiring-bias-project/data/train/train.csv


In [3]:
import os
os.makedirs("code", exist_ok=True)

In [4]:
%%writefile code/preprocessing.py
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

BASE_DIR = "/opt/ml/processing"
INPUT_DIR = os.path.join(BASE_DIR, "input")
TRAIN_OUTPUT_DIR = os.path.join(BASE_DIR, "train")
VALIDATION_OUTPUT_DIR = os.path.join(BASE_DIR, "validation")
TEST_OUTPUT_DIR = os.path.join(BASE_DIR, "test")

NUMERIC_FEATURES = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
CATEGORICAL_FEATURES = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]
TARGET = "income_binary"
PROTECTED_ATTRS = ["race", "sex", "age"]


def load_split(filename):
    return pd.read_csv(os.path.join(INPUT_DIR, filename))


def main():
    train_df = load_split("train.csv")
    val_df = load_split("validation.csv")
    test_df = load_split("test.csv")

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ]
    )

    X_train = preprocessor.fit_transform(train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
    X_val = preprocessor.transform(val_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
    X_test = preprocessor.transform(test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])

    y_train = train_df[TARGET].values
    y_val = val_df[TARGET].values
    y_test = test_df[TARGET].values

    train_out = pd.DataFrame(np.column_stack([y_train, X_train]))
    val_out = pd.DataFrame(np.column_stack([y_val, X_val]))

    test_out = pd.DataFrame(np.column_stack([y_test, X_test]))
    for attr in PROTECTED_ATTRS:
        test_out[f"_protected_{attr}"] = test_df[attr].values

    os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
    os.makedirs(VALIDATION_OUTPUT_DIR, exist_ok=True)
    os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

    train_out.to_csv(os.path.join(TRAIN_OUTPUT_DIR, "train.csv"), header=False, index=False)
    val_out.to_csv(os.path.join(VALIDATION_OUTPUT_DIR, "validation.csv"), header=False, index=False)
    test_out.to_csv(os.path.join(TEST_OUTPUT_DIR, "test.csv"), header=False, index=False)

    print("train shape:", train_out.shape)
    print("val shape:", val_out.shape)
    print("test shape:", test_out.shape)


if __name__ == "__main__":
    main()

Overwriting code/preprocessing.py


In [5]:
%%writefile code/evaluation.py
import os
import json
import tarfile
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

BASE_DIR = "/opt/ml/processing"
MODEL_PATH = os.path.join(BASE_DIR, "model", "model.tar.gz")
TEST_PATH = os.path.join(BASE_DIR, "test", "test.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "evaluation")

PROTECTED_COLS = ["_protected_race", "_protected_sex", "_protected_age"]


def disparate_impact_and_spd(y_pred, group_series, privileged_value):
    privileged_mask = group_series == privileged_value
    unprivileged_mask = ~privileged_mask

    if privileged_mask.sum() == 0 or unprivileged_mask.sum() == 0:
        return None, None

    p_privileged = y_pred[privileged_mask].mean()
    p_unprivileged = y_pred[unprivileged_mask].mean()

    di = None if p_privileged == 0 else float(p_unprivileged / p_privileged)
    spd = float(p_unprivileged - p_privileged)
    return di, spd


def main():
    with tarfile.open(MODEL_PATH) as tar:
        tar.extractall(path=".")

    model = xgb.Booster()
    model.load_model("xgboost-model")

    df = pd.read_csv(TEST_PATH, header=None)

    n_protected = len(PROTECTED_COLS)
    protected_df = df.iloc[:, -n_protected:].copy()
    protected_df.columns = PROTECTED_COLS

    feature_df = df.iloc[:, :-n_protected]
    y_true = feature_df.iloc[:, 0].values
    X_test = feature_df.iloc[:, 1:]

    dtest = xgb.DMatrix(X_test.values)
    y_pred_proba = model.predict(dtest)
    y_pred = (y_pred_proba >= 0.5).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_pred_proba)

    sex_di, sex_spd = disparate_impact_and_spd(y_pred, protected_df["_protected_sex"], "Male")
    race_di, race_spd = disparate_impact_and_spd(y_pred, protected_df["_protected_race"], "White")

    age_median = protected_df["_protected_age"].median()
    age_group = np.where(protected_df["_protected_age"] >= age_median, "older", "younger")
    age_di, age_spd = disparate_impact_and_spd(y_pred, pd.Series(age_group), "older")

    report = {
        "regression_metrics": {
            "accuracy": {"value": float(accuracy), "standard_deviation": "NaN"},
        },
        "classification_metrics": {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "auc": float(auc),
        },
        "fairness_metrics": {
            "sex": {"disparate_impact": sex_di, "statistical_parity_difference": sex_spd, "privileged_group": "Male"},
            "race": {"disparate_impact": race_di, "statistical_parity_difference": race_spd, "privileged_group": "White"},
            "age": {"disparate_impact": age_di, "statistical_parity_difference": age_spd, "privileged_group": "older (>= median age)"},
        },
        "fairness_gate": {
            "disparate_impact_sex": sex_di,
        },
    }

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(os.path.join(OUTPUT_DIR, "evaluation.json"), "w") as f:
        json.dump(report, f, indent=2)

    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

Overwriting code/evaluation.py


In [6]:
processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value="ml.t3.medium")

model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")

input_train_data = ParameterString(name="InputTrainData", default_value=train_s3)
input_validation_data = ParameterString(name="InputValidationData", default_value=validation_s3)
input_test_data = ParameterString(name="InputTestData", default_value=test_s3)

accuracy_threshold = ParameterFloat(name="AccuracyThreshold", default_value=0.75)
min_disparate_impact = ParameterFloat(name="MinDisparateImpact", default_value=0.80)
max_disparate_impact = ParameterFloat(name="MaxDisparateImpact", default_value=1.25)

batch_data = ParameterString(name="BatchData", default_value=test_s3)

In [7]:
consolidated_prefix = f"{prefix}/data/pipeline-input"

for split_name in ["train", "validation", "test"]:
    copy_source = {"Bucket": bucket, "Key": f"{prefix}/data/{split_name}/{split_name}.csv"}
    s3_client.copy_object(
        Bucket=bucket,
        CopySource=copy_source,
        Key=f"{consolidated_prefix}/{split_name}.csv",
    )

consolidated_input_s3 = f"s3://{bucket}/{consolidated_prefix}"
print(consolidated_input_s3)

s3://hiring-bias-adult-income-mustafa/hiring-bias-project/data/pipeline-input


In [8]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name="hiring-bias-process",
    sagemaker_session=pipeline_session,
    role=role,
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(source=consolidated_input_s3, destination="/opt/ml/processing/input"),
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
    code="code/preprocessing.py",
)

step_process = ProcessingStep(name="HiringBiasProcess", step_args=processor_args)

The input argument instance_type of function (sagemaker.image_uris.retrieve) is a pipeline variable (<class 'sagemaker.workflow.parameters.ParameterString'>), which is interpreted in pipeline execution time only. As the function needs to evaluate the argument value in SDK compile time, the default_value of this Parameter object will be used to override it. Please make sure the default_value is valid.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [9]:
# training job quota is 0 across every instance type on this account.
# request is still pending, so training locally in this kernel instead and
# packaging the model the same way a real training job would, so the rest
# of the pipeline (eval, registration, etc) doesn't need to change

import pandas as pd
import numpy as np
import xgboost as xgb
import tarfile
import os

os.makedirs("local_processing/input", exist_ok=True)
os.makedirs("local_processing/test", exist_ok=True)

s3_client.download_file(bucket, f"{consolidated_prefix}/train.csv", "local_processing/input/train.csv")
s3_client.download_file(bucket, f"{consolidated_prefix}/validation.csv", "local_processing/input/validation.csv")
s3_client.download_file(bucket, f"{consolidated_prefix}/test.csv", "local_processing/input/test.csv")

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

NUMERIC_FEATURES = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
CATEGORICAL_FEATURES = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]
TARGET = "income_binary"
PROTECTED_ATTRS = ["race", "sex", "age"]

train_df = pd.read_csv("local_processing/input/train.csv")
val_df = pd.read_csv("local_processing/input/validation.csv")
test_df = pd.read_csv("local_processing/input/test.csv")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

X_train = preprocessor.fit_transform(train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
X_val = preprocessor.transform(val_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
X_test = preprocessor.transform(test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])

y_train = train_df[TARGET].values
y_val = val_df[TARGET].values
y_test = test_df[TARGET].values

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

params = {
    "objective": "binary:logistic",
    "max_depth": 5,
    "eta": 0.2,
    "subsample": 0.8,
    "eval_metric": "auc",
}

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=100,
    evals=[(dtrain, "train"), (dval, "validation")],
    verbose_eval=10,
)

bst.save_model("xgboost-model")

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("xgboost-model")

model_s3_key = f"{prefix}/models/local-train/model.tar.gz"
s3_client.upload_file("model.tar.gz", bucket, model_s3_key)
local_model_artifact_s3 = f"s3://{bucket}/{model_s3_key}"
print("model uploaded:", local_model_artifact_s3)

test_out = pd.DataFrame(np.column_stack([y_test, X_test]))
for attr in PROTECTED_ATTRS:
    test_out[f"_protected_{attr}"] = test_df[attr].values

test_out.to_csv("local_processing/test/test.csv", header=False, index=False)
local_test_s3_key = f"{prefix}/data/local-processed/test.csv"
s3_client.upload_file("local_processing/test/test.csv", bucket, local_test_s3_key)
local_test_s3 = f"s3://{bucket}/{local_test_s3_key}"
print("test data uploaded:", local_test_s3)

[0]	train-auc:0.88669	validation-auc:0.88225


[10]	train-auc:0.91606	validation-auc:0.90967


[20]	train-auc:0.92472	validation-auc:0.91514


[30]	train-auc:0.93062	validation-auc:0.91897


[40]	train-auc:0.93438	validation-auc:0.92136


...

[90]	train-auc:0.94522	validation-auc:0.92371


[99]	train-auc:0.94645	validation-auc:0.92384


model uploaded: s3://hiring-bias-adult-income-mustafa/hiring-bias-project/models/local-train/model.tar.gz


/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [07:18:10] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


test data uploaded: s3://hiring-bias-adult-income-mustafa/hiring-bias-project/data/local-processed/test.csv


In [10]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile
from sagemaker.image_uris import retrieve

# need an xgboost container image to run the evaluation script in (it loads
# the model with xgb.Booster, so the container needs xgboost installed)
image_uri = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.t3.medium",
)

script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="hiring-bias-eval",
    sagemaker_session=pipeline_session,

SyntaxError: incomplete input (3356025188.py, line 21)

In [11]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile
from sagemaker.image_uris import retrieve

# need an xgboost container image to run the evaluation script in (it loads
# the model with xgb.Booster, so the container needs xgboost installed)
image_uri = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.t3.medium",
)

script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="hiring-bias-eval",
    sagemaker_session=pipeline_session,
    role=role,
)

evaluation_report = PropertyFile(
    name="HiringBiasEvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=local_model_artifact_s3,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=local_test_s3,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="code/evaluation.py",
)

step_eval = ProcessingStep(
    name="HiringBiasEvaluate",
    step_args=eval_args,
    property_files=[evaluation_report],
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [12]:
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep

model = Model(
    image_uri=image_uri,
    model_data=local_model_artifact_s3,
    sagemaker_session=pipeline_session,
    role=role,
)

create_model_args = model.create(instance_type="ml.m5.large")
step_create_model = ModelStep(name="HiringBiasCreateModel", step_args=create_model_args)

In [13]:
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/batch-transform-output",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="HiringBiasTransform",
    step_args=transformer.transform(
        data=batch_data,
        content_type="text/csv",
    ),
)

In [14]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)

step_register = ModelStep(name="HiringBiasRegisterModel", step_args=register_args)

In [15]:
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

step_fail = FailStep(
    name="HiringBiasModelFails",
    error_message=Join(
        on=" ",
        values=[
            "Model did not pass the CI gate. Required: accuracy >=",
            accuracy_threshold,
            "AND disparate impact (sex) between",
            min_disparate_impact,
            "and",
            max_disparate_impact,
            ". Check evaluation.json for actual values.",
        ],
    ),
)

In [16]:
from sagemaker.workflow.conditions import (
    ConditionGreaterThanOrEqualTo,
    ConditionLessThanOrEqualTo,
)
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

cond_accuracy = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.accuracy",
    ),
    right=accuracy_threshold,
)

cond_di_lower = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="fairness_gate.disparate_impact_sex",
    ),
    right=min_disparate_impact,
)

cond_di_upper = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="fairness_gate.disparate_impact_sex",
    ),
    right=max_disparate_impact,
)

step_cond = ConditionStep(
    name="CheckAccuracyAndFairness",
    conditions=[cond_accuracy, cond_di_lower, cond_di_upper],
    if_steps=[step_create_model, step_register, step_transform],
    else_steps=[step_fail],
)

In [17]:
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        processing_instance_type,
        model_approval_status,
        input_train_data,
        input_validation_data,
        input_test_data,
        accuracy_threshold,
        min_disparate_impact,
        max_disparate_impact,
        batch_data,
    ],
    steps=[step_process, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)
print("pipeline created/updated:", pipeline_name)

pipeline created/updated: HiringBiasCICDPipeline


In [18]:
execution_success = pipeline.start()
print("started execution:", execution_success.arn)

started execution: arn:aws:sagemaker:us-east-1:498374411846:pipeline/HiringBiasCICDPipeline/execution/w7edqoo3z3yy


In [19]:
execution_success.wait()

for step in execution_success.list_steps():
    print(step["StepName"], "-", step["StepStatus"])
    if "FailureReason" in step:
        print("   reason:", step["FailureReason"])

WaiterError: Waiter PipelineExecutionComplete failed: Waiter encountered a terminal failure state: For expression "PipelineExecutionStatus" we matched expected path: "Failed"

In [20]:
for step in execution_success.list_steps():
    print(step["StepName"], "-", step["StepStatus"])
    if "FailureReason" in step:
        print("   reason:", step["FailureReason"])
        

HiringBiasModelFails - Failed
   reason: Model did not pass the CI gate. Required: accuracy >= 0.75 AND disparate impact (sex) between 0.8 and 1.25 . Check evaluation.json for actual values.
CheckAccuracyAndFairness - Succeeded
HiringBiasProcess - Succeeded
HiringBiasEvaluate - Succeeded


In [21]:
import json

eval_s3_uri = step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
print(eval_s3_uri)

!aws s3 cp {eval_s3_uri}/evaluation.json ./evaluation.json --quiet
with open("evaluation.json") as f:
    report = json.load(f)
print(json.dumps(report, indent=2))

s3://sagemaker-us-east-1-498374411846/hiring-bias-eval-2026-06-20-07-19-27-270/output/evaluation


{
  "regression_metrics": {
    "accuracy": {
      "value": 0.8677771226415094,
      "standard_deviation": "NaN"
    }
  },
  "classification_metrics": {
    "accuracy": 0.8677771226415094,
    "precision": 0.7824207492795389,
    "recall": 0.646044021415824,
    "f1": 0.70772238514174,
    "auc": 0.9240503451621171
  },
  "fairness_metrics": {
    "sex": {
      "disparate_impact": 0.3125114414111172,
      "statistical_parity_difference": -0.18234844497907426,
      "privileged_group": "Male"
    },
    "race": {
      "disparate_impact": 0.5388118203128003,
      "statistical_parity_difference": -0.10110847372666633,
      "privileged_group": "White"
    },
    "age": {
      "disparate_impact": 0.28970184341834804,
      "statistical_parity_difference": -0.2220828861663563,
      "privileged_group": "older (>= median age)"
    }
  },
  "fairness_gate": {
    "disparate_impact_sex": 0.3125114414111172
  }
}


In [22]:
execution_demo = pipeline.start(
    parameters={
        "MinDisparateImpact": 0.0,
        "MaxDisparateImpact": 999.0,
    }
)
print("started execution:", execution_demo.arn)

started execution: arn:aws:sagemaker:us-east-1:498374411846:pipeline/HiringBiasCICDPipeline/execution/udelcdemu977


In [23]:
execution_demo.wait()

for step in execution_demo.list_steps():
    print(step["StepName"], "-", step["StepStatus"])
    if "FailureReason" in step:
        print("   reason:", step["FailureReason"])

WaiterError: Waiter PipelineExecutionComplete failed: Waiter encountered a terminal failure state: For expression "PipelineExecutionStatus" we matched expected path: "Failed"

In [24]:
for step in execution_demo.list_steps():
    print(step["StepName"], "-", step["StepStatus"])
    if "FailureReason" in step:
        print("   reason:", step["FailureReason"])

HiringBiasTransform - Failed
   reason: Step failed after exhausting 1 retry attempt(s):
Attempt 1 error: ClientError: Failed to invoke sagemaker:CreateTransformJob. Error Details: The account-level service limit 'ml.m5.large for transform job usage' is 0 Instances, with current utilization of 0 Instances and a request delta of 1 Instances. Please use AWS Service Quotas to request an increase for this quota. If AWS Service Quotas is not available, contact AWS support to request an increase for this quota.
Retry not appropriate on execution of step with PipelineExecutionArn arn:aws:sagemaker:us-east-1:498374411846:pipeline/hiringbiascicdpipeline/execution/udelcdemu977 and StepId HiringBiasTransform. No retry policy configured for the exception type SAGEMAKER_RESOURCE_LIMIT.
HiringBiasCreateModel-CreateModel - Succeeded
HiringBiasRegisterModel-RegisterModel - Succeeded
CheckAccuracyAndFairness - Succeeded
HiringBiasProcess - Succeeded
HiringBiasEvaluate - Succeeded


In [25]:
# transform job quota is also stuck at 0 and the request hasn't cleared yet,
# same situation as training. running batch inference locally here instead,
# using the model we already trained and uploaded, writing predictions to
# the same s3 output path the managed transform step would have used

import pandas as pd
import xgboost as xgb

# pull the model back down (we already have it locally from cell 10, but
# re-loading from s3 to mirror exactly what a real transform job would do)
s3_client.download_file(bucket, model_s3_key, "model_for_inference.tar.gz")

import tarfile
with tarfile.open("model_for_inference.tar.gz") as tar:
    tar.extractall(path="inference_model")

bst_inference = xgb.Booster()
bst_inference.load_model("inference_model/xgboost-model")

# load the same processed test set the transform step would have scored
s3_client.download_file(bucket, local_test_s3_key, "local_processing/test/test_for_inference.csv")
test_for_inference = pd.read_csv("local_processing/test/test_for_inference.csv", header=None)

# strip off the label column and the protected attribute columns tacked on
# at the end - just want the raw feature columns for scoring
n_protected = 3
feature_only = test_for_inference.iloc[:, 1:-n_protected]

dtest_inference = xgb.DMatrix(feature_only.values)
predictions = bst_inference.predict(dtest_inference)

predictions_df = pd.DataFrame({"prediction": predictions})
predictions_df.to_csv("batch_predictions.csv", index=False, header=False)

# upload to the same output prefix the transformer was configured to use
transform_output_key = f"{prefix}/batch-transform-output/test.csv.out"
s3_client.upload_file("batch_predictions.csv", bucket, transform_output_key)

batch_output_s3 = f"s3://{bucket}/{transform_output_key}"
print("batch predictions uploaded to:", batch_output_s3)
print("sample predictions:")
print(predictions_df.head(10))

/tmp/ipykernel_13942/2326643704.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="inference_model")


batch predictions uploaded to: s3://hiring-bias-adult-income-mustafa/hiring-bias-project/batch-transform-output/test.csv.out
sample predictions:
   prediction
0    0.027967
1    0.002821
2    0.623832
3    0.649871
4    0.999030
5    0.668182
6    0.006323
7    0.008858
8    0.013323
9    0.162863


In [26]:
# model monitor needs a baseline to compare future live traffic against -
# using the training data we already have in s3 for this

from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.t3.medium",
    volume_size_in_gb=10,
    max_runtime_in_seconds=1800,
)

baseline_results_uri = f"s3://{bucket}/{prefix}/model-monitor/baseline"

monitor.suggest_baseline(
    baseline_dataset=consolidated_input_s3 + "/train.csv",
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=baseline_results_uri,
    wait=True,
)

print("baseline written to:", baseline_results_uri)

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: .


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-20-07-56-45-438


.

.

...

2026-06-20 08:01:42,192 INFO Main: Start analyzing with args: --analytics_input /tmp/spark_job_config.json
2026-06-20 08:01:42,289 INFO Main: Analytics input path: DataAnalyzerParams(/tmp/spark_job_config.json,yarn)
2026-06-20 08:01:42,314 INFO FileUtil: Read file from path /tmp/spark_job_config.json.
2026-06-20 08:01:43,175 INFO spark.SparkContext: Running Spark version 3.3.0
2026-06-20 08:01:43,213 INFO resource.ResourceUtils: ==============================================================
2026-06-20 08:01:43,214 INFO resource.ResourceUtils: No custom resources configured for spark.driver.
2026-06-20 08:01:43,214 INFO resource.ResourceUtils: ==============================================================
2026-06-20 08:01:43,215 INFO spark.SparkContext: Submitted application: SageMakerDataAnalyzer
2026-06-20 08:01:43,277 INFO resource.ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amo

2026-06-20 08:01:52,421 INFO yarn.Client: Uploading resource file:/tmp/spark-93a7a97e-eca0-4131-81bb-3511c1f9ac7b/__spark_conf__4516641143399981679.zip -> hdfs://10.2.105.166/user/root/.sparkStaging/application_1781942484808_0001/__spark_conf__.zip
2026-06-20 08:01:52,516 INFO spark.SecurityManager: Changing view acls to: root
2026-06-20 08:01:52,517 INFO spark.SecurityManager: Changing modify acls to: root
2026-06-20 08:01:52,517 INFO spark.SecurityManager: Changing view acls groups to: 
2026-06-20 08:01:52,517 INFO spark.SecurityManager: Changing modify acls groups to: 
2026-06-20 08:01:52,518 INFO spark.SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users  with view permissions: Set(root); groups with view permissions: Set(); users  with modify permissions: Set(root); groups with modify permissions: Set()
2026-06-20 08:01:52,568 INFO yarn.Client: Submitting application application_1781942484808_0001 to ResourceManager
2026-06-20 08:01:52,954 INFO impl.Y

2026-06-20 08:02:03,017 INFO yarn.Client: Application report for application_1781942484808_0001 (state: ACCEPTED)
2026-06-20 08:02:04,023 INFO yarn.Client: Application report for application_1781942484808_0001 (state: RUNNING)
2026-06-20 08:02:04,024 INFO yarn.Client: 
#011 client token: N/A
#011 diagnostics: N/A
#011 ApplicationMaster host: 10.2.105.166
#011 ApplicationMaster RPC port: -1
#011 queue: default
#011 start time: 1781942512748
#011 final status: UNDEFINED
#011 tracking URL: http://algo-1:8088/proxy/application_1781942484808_0001/
#011 user: root
2026-06-20 08:02:04,026 INFO cluster.YarnClientSchedulerBackend: Application application_1781942484808_0001 has started running.
2026-06-20 08:02:04,078 INFO util.Utils: Successfully started service 'org.apache.spark.network.netty.NettyBlockTransferService' on port 39461.
2026-06-20 08:02:04,078 INFO netty.NettyBlockTransferService: Server created on 10.2.105.166:39461
2026-06-20 08:02:04,083 INFO storage.BlockManager: Using org.ap

2026-06-20 08:02:25,442 INFO cluster.YarnClientSchedulerBackend: SchedulerBackend is ready for scheduling beginning after waiting maxRegisteredResourcesWaitingTime: 30000000000(ns)


baseline written to: s3://hiring-bias-adult-income-mustafa/hiring-bias-project/model-monitor/baseline


In [27]:
import json

baseline_job = monitor.latest_baselining_job
constraints = baseline_job.suggested_constraints().body_dict

print("sample of suggested constraints:")
print(json.dumps(constraints["features"][:3], indent=2))

Could not retrieve constraints file at location 's3://hiring-bias-adult-income-mustafa/hiring-bias-project/model-monitor/baseline/constraints.json'. To manually retrieve Constraints object from a given uri, use 'my_model_monitor.constraints(my_s3_uri)' or 'Constraints.from_s3_uri(my_s3_uri)'


UnexpectedStatusException: The underlying job is not in 'Completed' state. You may only retrieve files for a job that has completed successfully.